In [0]:
#%pip install faiss-cpu sentence-transformers
#%restart_python

 
#%pip install torch torchvision torchaudio  
#%restart_python
 

In [0]:
# Assuming your source Delta table is named "main.default.numerical_data"
from pyspark.sql.functions import udf, concat_ws, col
from pyspark.sql.types import ArrayType, FloatType

table_name = "workspace.default.sales_data_bronze"
df = spark.read.table(table_name)
#string_list=df.select('combined_text').collect()
#print (string_list[0])
df = df.filter((col("COUNTRY") == "Canada") )
#display(df)


In [0]:
 
%sql 
 
 SHOW VOLUMES IN default

database,volume_name


In [0]:
from sentence_transformers import SentenceTransformer
from pyspark.sql.functions import udf, concat_ws, col
from pyspark.sql.types import ArrayType, FloatType
from pyspark.sql.functions import lit
 

model = SentenceTransformer('all-MiniLM-L6-v2') 
 
def get_vector(text):
    return model.encode(text)

embeddings_udf = udf(get_vector, ArrayType(FloatType()))
#df_embeddings = df.withColumn('embeddings', embeddings_udf(concat_ws(' ', col('combined_text'))))
for x in df.select('combined_text').collect():
  df = df.withColumn("embeddings", lit(model.encode(x).tolist()))

#df = df.withColumn("embeddings", df["embeddings"].cast(ArrayType(FloatType()))) 
#display(df)
df.write.format("delta").mode("overwrite").saveAsTable("sales_data_embeddings")



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
